###  Weather Data Collection (Daily)

This notebook collects daily weather data (2007–2026) for all Japanese prefectures using the Open-Meteo API.

### Features:
- Multi-year data collection
- Prefecture-wise batching
- Automatic CSV storage
- Basic fault tolerance

### Output:
data/weather_daily/*.csv


In [ ]:
import pandas as pd
import requests
import time
import os

In [ ]:
BASE_DIR = "data"
RAW_WEATHER_DIR = os.path.join(BASE_DIR, "weather_daily_new_2026")

os.makedirs(RAW_WEATHER_DIR, exist_ok=True)

RAW_WEATHER_DIR


In [ ]:
# Setting the lat and lon for each prefecture

PREFECTURE_CITIES = {
    "Hokkaido":       {"city": "Sapporo",     "lat": 43.0618, "lon": 141.3545},
    "Aomori":         {"city": "Aomori",      "lat": 40.8222, "lon": 140.7474},
    "Iwate":          {"city": "Morioka",     "lat": 39.7036, "lon": 141.1527},
    "Miyagi":         {"city": "Sendai",      "lat": 38.2682, "lon": 140.8694},
    "Akita":          {"city": "Akita",       "lat": 39.7200, "lon": 140.1025},
    "Yamagata":       {"city": "Yamagata",    "lat": 38.2554, "lon": 140.3396},
    "Fukushima":      {"city": "Fukushima",   "lat": 37.7503, "lon": 140.4676},
    "Ibaraki":        {"city": "Mito",        "lat": 36.3659, "lon": 140.4710},
    "Tochigi":        {"city": "Nikko",       "lat": 36.7580, "lon": 139.5980},
    "Gunma":          {"city": "Kusatsu",     "lat": 36.6228, "lon": 138.5963},
    "Saitama":        {"city": "Kawagoe",     "lat": 35.9251, "lon": 139.4858},
    "Chiba":          {"city": "Chiba",       "lat": 35.6073, "lon": 140.1063},
    "Tokyo":          {"city": "Tokyo",       "lat": 35.6895, "lon": 139.6917},
    "Kanagawa":       {"city": "Yokohama",    "lat": 35.4437, "lon": 139.6380},
    "Niigata":        {"city": "Niigata",     "lat": 37.9161, "lon": 139.0364},
    "Toyama":         {"city": "Toyama",      "lat": 36.6953, "lon": 137.2113},
    "Ishikawa":       {"city": "Kanazawa",    "lat": 36.5613, "lon": 136.6562},
    "Fukui":          {"city": "Fukui",       "lat": 36.0652, "lon": 136.2216},
    "Yamanashi":      {"city": "Kawaguchiko", "lat": 35.4993, "lon": 138.7686},
    "Nagano":         {"city": "Nagano",      "lat": 36.6486, "lon": 138.1948},
    "Gifu":           {"city": "Takayama",    "lat": 36.1463, "lon": 137.2510},
    "Shizuoka":       {"city": "Shizuoka",    "lat": 34.9756, "lon": 138.3828},
    "Aichi":          {"city": "Nagoya",      "lat": 35.1815, "lon": 136.9066},
    "Mie":            {"city": "Ise",         "lat": 34.4885, "lon": 136.7093},
    "Shiga":          {"city": "Otsu",        "lat": 35.0170, "lon": 135.8540},
    "Kyoto":          {"city": "Kyoto",       "lat": 35.0116, "lon": 135.7681},
    "Osaka":          {"city": "Osaka",       "lat": 34.6937, "lon": 135.5023},
    "Hyogo":          {"city": "Kobe",        "lat": 34.6901, "lon": 135.1955},
    "Nara":           {"city": "Nara",        "lat": 34.6851, "lon": 135.8048},
    "Wakayama":       {"city": "Wakayama",    "lat": 34.2261, "lon": 135.1675},
    "Tottori":        {"city": "Tottori",     "lat": 35.5011, "lon": 134.2351},
    "Shimane":        {"city": "Matsue",      "lat": 35.4723, "lon": 133.0505},
    "Okayama":        {"city": "Okayama",     "lat": 34.6551, "lon": 133.9195},
    "Hiroshima":      {"city": "Hiroshima",   "lat": 34.3853, "lon": 132.4553},
    "Yamaguchi":      {"city": "Yamaguchi",   "lat": 34.1785, "lon": 131.4737},
    "Tokushima":      {"city": "Tokushima",   "lat": 34.0658, "lon": 134.5594},
    "Kagawa":         {"city": "Takamatsu",   "lat": 34.3428, "lon": 134.0466},
    "Ehime":          {"city": "Matsuyama",   "lat": 33.8394, "lon": 132.7654},
    "Kochi":          {"city": "Kochi",       "lat": 33.5597, "lon": 133.5311},
    "Fukuoka":        {"city": "Fukuoka",     "lat": 33.5904, "lon": 130.4017},
    "Saga":           {"city": "Saga",        "lat": 33.2635, "lon": 130.3009},
    "Nagasaki":       {"city": "Nagasaki",    "lat": 32.7503, "lon": 129.8777},
    "Kumamoto":       {"city": "Kumamoto",    "lat": 32.8031, "lon": 130.7079},
    "Oita":           {"city": "Beppu",       "lat": 33.2795, "lon": 131.4978},
    "Miyazaki":       {"city": "Miyazaki",    "lat": 31.9077, "lon": 131.4202},
    "Kagoshima":      {"city": "Kagoshima",   "lat": 31.5966, "lon": 130.5571},
    "Okinawa":        {"city": "Naha",        "lat": 26.2124, "lon": 127.6809},
}


In [ ]:
# fetching the daily weather for each prefecture 

def fetch_daily_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
        ],
        "timezone": "Asia/Tokyo"
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        return pd.DataFrame(data["daily"])
    
    except Exception as e:
        print(f"[ERROR] {e}")
        return None


In [ ]:
def collect_all_prefectures(prefecture_map, start_year=2026, end_year=2026, delay_sec=3):
    for prefecture_name, info in prefecture_map.items():

        output_path = os.path.join(RAW_WEATHER_DIR, f"{prefecture_name}_daily.csv")

        if os.path.exists(output_path):
            print(f"[SKIP] {prefecture_name}")
            continue

        print(f"\n[INFO] Collecting {prefecture_name} ({info['city']})")

        all_years = []

        for year in range(start_year, end_year + 1):
            print(f"  → {year}")

            df_year = fetch_daily_weather(
                lat=info["lat"],
                lon=info["lon"],
                start_date=f"{year}-01-01",
                end_date=f"{year}-06-31"
            )

            if df_year is None:
                print(f"[WARNING] Skipping {year}")
                continue

            df_year["prefecture"] = prefecture_name
            df_year["year"] = year

            all_years.append(df_year)
            time.sleep(delay_sec)

        if all_years:
            df_all = pd.concat(all_years, ignore_index=True)
            df_all.to_csv(output_path, index=False)
            print(f"[SAVED] {output_path}")

In [ ]:
collect_all_prefectures(PREFECTURE_CITIES, delay_sec=4)


In [ ]:
import pandas as pd
import glob
import os

# Folder containing all weather CSV files
folder_path = "data/weather_daily_new"

# Get all CSV files
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

for file in csv_files:
    # Read CSV
    df = pd.read_csv(file)

    # Convert year column to numeric
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    # Keep only data from 2007 onwards
    df = df[df["year"] >= 2007]

    # Save back to the same file
    df.to_csv(file, index=False)

    print(f"Processed: {os.path.basename(file)}")

print("✅ All files processed successfully!")